<a href="https://colab.research.google.com/github/oliviadellaglio/ds2002-fa26/blob/main/01-foundations/2026_09_23_%E2%80%94_Cleaning_Clinic_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [ ]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [ ]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [ ]:
# TODO

shape = df.shape
dtypes = df.dtypes
is_null = df.isnull().sum()
dupes = df.duplicated().sum()

print('Shape:', shape)
print('Dtypes: ', dtypes)
print('Null Count: ', is_null)
print("Number of duplicate rows: ", dupes)

Shape: (8, 6)
Dtypes:  order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object
Null Count:  order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
Number of duplicate rows:  1


**What is wrong with this data?** List at least five specific problems:

1. Category names inconsistant
2. Item names inconsistant
3. Some prices have dollar signs, others do not
4. Time/dates in different formats
5. Negative quantity

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [ ]:
removed = df.duplicated().sum()   # TODO: how many duplicates were there?
clean =   df.drop_duplicates().copy()  # TODO: df with duplicates dropped, copied

log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [ ]:
clean['price'] = (
    clean['price']
    .astype(str)
    .str.strip()
    .str.replace('$', '', regex=False)
    .astype(float)
)

assert clean['price'].dtype == 'float64'

### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [ ]:

clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()
negative = (clean['qty'] < 0).sum()

clean = clean.dropna(subset=['qty']).copy()
log('quantity_missing', 'dropped rows with missing quantity', missing)

log('quantity_negative', 'kept negative quantities as refunds', negative)


[quantity_missing] dropped rows with missing quantity (1 row(s))
[quantity_negative] kept negative quantities as refunds (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [ ]:
print('before:', sorted(clean['category'].unique()))


clean['category'] = (
    clean['category']
    .str.lower()
    .str.strip()
    .str.replace(r'[^a-z0-9]', '', regex=True)
)


CATEGORY_MAP = {
    'food': 'Food',
    'merch': 'Merch',
    'apparel': 'Apparel',
    'raingear': 'RainGear'
}

clean['category'] = clean['category'].map(CATEGORY_MAP)

print('after: ', sorted(clean['category'].unique()))

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['Apparel', 'Food', 'Merch', 'RainGear']


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [ ]:
print('before:', sorted(clean['item'].dropna().unique()))

clean['item'] = (
    clean['item']
    .str.lower()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

ITEM_MAP = {
    'cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'foam finger': 'Foam Finger',
    'uva t-shirt': 'UVA T-Shirt',
    'rain poncho': 'Rain Poncho'
}

clean['item'] = clean['item'].map(ITEM_MAP)

missing_item = clean['item'].isna().sum()
clean = clean.dropna(subset=['item']).copy()

log('item_missing', 'dropped rows with missing item', missing_item)

print('after: ', sorted(clean['item'].unique()))

before: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
[item_missing] dropped rows with missing item (1 row(s))
after:  ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt']


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [ ]:
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce')

failed = clean['ts'].isna().sum()

print('Failed timestamp parses:', failed)

clean['hour'] = clean['ts'].dt.hour

Failed timestamp parses: 3


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [ ]:
assert clean['price'].dtype == 'float64'
assert pd.api.types.is_numeric_dtype(clean['qty'])
assert not clean['item'].isna().any()
assert not clean['category'].isna().any()
assert clean['ts'].dtype == 'datetime64[ns]'

clean['revenue'] = clean['qty'] * clean['price']


print('rows:', len(clean))
print('units:', clean['qty'].sum())
print('revenue:', clean['revenue'].sum())
print('distinct categories:', clean['category'].nunique())

rows: 5
units: 6.0
revenue: 76.5
distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [ ]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,quantity_missing,dropped rows with missing quantity,1
2,quantity_negative,kept negative quantities as refunds,1
3,item_missing,dropped rows with missing item,1


**The decision that mattered most:**

>The decision that mattered the most was to include the refund. The quanitiy was -3, and the cost was $6, so it decreased revenue by $18



**Revenue with it:** $76.50  
**Revenue without it:** $94.50

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [ ]:
# Checkpoint
rows_after = len(clean)
revenue_after = clean['revenue'].sum()
biggest_decision = 'Including the refund with the revenue'
revenue_other_way = clean.loc[clean['qty'] >= 0, 'revenue'].sum()

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 76.5
decision that mattered: Including the refund with the revenue
revenue the other way: 94.5
